# 04 · Connecting the application to data and tools

**Session 01 · notebook 4 of 6**

## Objective

Give the model a tool that returns authoritative statistics, let it call the tool, and make its output a typed object that other software can read. See what happens when the data does not exist.

## The application after this notebook

```
question ──> model ──┬──> tool call: get_indicator(indicator, country, years) ──> data ──┐
                     │<──────────────────────────────────────────────────────────────────┘
                     └──> answer, or a JSON object that matches a schema
```

Passages are for definitions and context. Numbers come from a tool over the data.

In [1]:
# Setup. In Colab this clones the repository and installs the dependencies; locally it uses the folder as is.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/avsolatorio/ai4data.git"   # replace with the published repository when released
BRANCH = "feat/training-series"
SESSION = "training/session-01-building-ai-applications/notebooks"

if not pathlib.Path("workshop.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, "/content/ai4data"], check=True)
    os.chdir(f"/content/ai4data/{SESSION}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "pydantic", "scikit-learn", "requests"], check=True)
sys.path.insert(0, os.getcwd())

# Optional: a key for an OpenAI-compatible API. Without one, the notebook uses recorded outputs and says so.
try:
    from google.colab import userdata  # type: ignore
    for name in ("WORKSHOP_API_KEY", "WORKSHOP_BASE_URL", "WORKSHOP_MODEL"):
        if userdata.get(name):
            os.environ[name] = userdata.get(name)
except Exception:
    pass

import workshop as w
llm = w.LLM()
print("Model client:", llm.mode())

Model client: recorded outputs (no WORKSHOP_API_KEY set)


## 1. A number from memory and a number from data

In [2]:
answer = llm.complete([{"role": "user", "content": "What was Rwanda's GDP per capita in 2023?"}], scenario="nb04_from_memory")
w.show(answer)
print()
r = w.get_indicator("NY.GDP.PCAP.CD", "Rwanda", start=2023, end=2023)
print("Data tool:", r["name"], r["observations"], "·", r["source"])

[recorded output: nb04_from_memory]
Rwanda's GDP per capita in 2023 was approximately 1,000 US dollars (current prices), according to
World Bank data. Please verify against the latest World Development Indicators, as figures are
revised.

Data tool: GDP per capita (current US$) [{'year': 2023, 'value': 1061.0579, 'claim_id': 'NY.GDP.PCAP.CD:RWA:2023'}] · World Development Indicators (Country official statistics)


The memory answer is close and hedged, with no way for a reader to tell how close. The data answer carries the indicator, the year, the value, the source, and a `claim_id` that notebook 06 will use to verify it.

## 2. The tool and its schema

A tool is a function plus a description the model can read. The description says what the tool is for and what its parameters mean; the model fills the parameters from the question.

In [3]:
import json
print(json.dumps(w.INDICATOR_TOOL, indent=2))
print()
print("available indicators:")
for i in w.list_indicators():
    print(f"  {i['id']:20s} {i['name']}")

{
  "type": "function",
  "function": {
    "name": "get_indicator",
    "description": "Observations of a World Development Indicators series for one country. Use it for any statistic; do not quote figures from memory.",
    "parameters": {
      "type": "object",
      "properties": {
        "indicator": {
          "type": "string",
          "description": "Indicator id, for example IT.NET.USER.ZS or NY.GDP.PCAP.CD"
        },
        "country": {
          "type": "string",
          "description": "Country name or ISO3 code"
        },
        "start": {
          "type": "integer",
          "description": "First year (optional)"
        },
        "end": {
          "type": "integer",
          "description": "Last year (optional)"
        }
      },
      "required": [
        "indicator",
        "country"
      ]
    }
  }
}

available indicators:
  NY.GDP.PCAP.CD       GDP per capita (current US$)
  IT.NET.USER.ZS       Individuals using the Internet (% of population)
  NY

## 3. The model calls the tool

`run_tool_loop` sends the question with the tool schema, runs the tool the model asks for, returns the result, and repeats until the model answers in text. The trace shows every call.

In [4]:
INSTRUCTIONS = """You are a research assistant for development data.
Use get_indicator for every statistic; do not quote figures from memory. Name the indicator and the source with each figure.
If the tool returns no observation for the year asked, say so and give the latest available year."""

handlers = {"get_indicator": w.get_indicator}

llm.reset()
final, trace = w.run_tool_loop(
    llm,
    [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": "What was Rwanda's GDP per capita in 2023?"}],
    tools=[w.INDICATOR_TOOL], handlers=handlers, scenario="nb04_tool_call",
)
print()
w.show(final)

step 1: get_indicator({"indicator": "NY.GDP.PCAP.CD", "country": "Rwanda", "start": 2020, "end": 2024}) -> {"indicator": "NY.GDP.PCAP.CD", "country": "RWA", "name": "GDP per capita (current US$)", "unit": "", "source": "World Development Indicators (Country official …

Rwanda's GDP per capita in 2023 was 1,061 US dollars in current prices (World Development
Indicators, NY.GDP.PCAP.CD). It rose from 803 dollars in 2020 to 1,007 in 2022 and 1,061 in 2023,
and was 1,060 in 2024.


## 4. Data that does not exist

The planted problem: a year with no observation. The tool returns an empty list, and the instructions say what to do with it.

In [5]:
llm.reset()
final, trace = w.run_tool_loop(
    llm,
    [{"role": "system", "content": INSTRUCTIONS}, {"role": "user", "content": "What share of Kenya's population used the internet in 2025?"}],
    tools=[w.INDICATOR_TOOL], handlers=handlers, scenario="nb04_missing_year",
)
print()
w.show(final)
print("\ntool result for 2025:", trace[0]["result"]["observations"])

step 1: get_indicator({"indicator": "IT.NET.USER.ZS", "country": "Kenya", "start": 2025, "end": 2025}) -> {"indicator": "IT.NET.USER.ZS", "country": "KEN", "name": "Individuals using the Internet (% of population)", "unit": "", "source": "World Development Indicator…

No observation is available for 2025. The series IT.NET.USER.ZS (Individuals using the Internet, %
of population) has no 2025 value for Kenya in the data returned. The latest available year is 2024,
when the value was 35.0 percent.

tool result for 2025: []


Compare with notebook 01, where the model answered the same question from a brief. Here the absence is a fact returned by the tool, which the application can test for.

## 5. Structured output

An answer that another program will read, store, or display needs a fixed shape. The schema below is a Pydantic model; its JSON schema is sent with the request, and the response is validated on the way back.

In [6]:
print(json.dumps(w.ComparisonAnswer.model_json_schema(), indent=2)[:1200], "…")

{
  "$defs": {
    "CountrySummary": {
      "properties": {
        "country": {
          "title": "Country",
          "type": "string"
        },
        "internet_latest": {
          "$ref": "#/$defs/Observation"
        },
        "gdp_per_capita_latest": {
          "$ref": "#/$defs/Observation"
        },
        "trend": {
          "description": "One or two sentences on the trend since 2010",
          "title": "Trend",
          "type": "string"
        }
      },
      "required": [
        "country",
        "internet_latest",
        "gdp_per_capita_latest",
        "trend"
      ],
      "title": "CountrySummary",
      "type": "object"
    },
    "Observation": {
      "properties": {
        "indicator": {
          "description": "Indicator id",
          "title": "Indicator",
          "type": "string"
        },
        "country": {
          "description": "ISO3 code",
          "title": "Country",
          "type": "string"
        },
        "year": {
         

In [7]:
schema = {"type": "json_schema", "json_schema": {"name": "comparison", "schema": w.ComparisonAnswer.model_json_schema()}}
completion = llm.complete(
    [
        {"role": "system", "content": "Fill the schema from the data you are given. Use the claim_id values exactly as given."},
        {"role": "user", "content": "DATA:\n" + json.dumps([w.get_indicator(i, c, start=2024) for i in ["IT.NET.USER.ZS", "NY.GDP.PCAP.CD"] for c in ["Kenya", "Rwanda"]]) + "\n\nQUESTION: " + w.MAIN_QUESTION},
    ],
    scenario="nb04_structured", response_format=schema,
)
parsed = w.ComparisonAnswer.model_validate_json(completion.content)
for cs in parsed.countries:
    print(cs.country, "internet", cs.internet_latest.year, cs.internet_latest.value, "| GDP pc", cs.gdp_per_capita_latest.year, round(cs.gdp_per_capita_latest.value))
print()
print(parsed.comparison)
print(parsed.sources)

KEN internet 2024 34.9762 | GDP pc 2024 2133
RWA internet 2024 31.7033 | GDP pc 2024 1060

Kenya has the higher level on both indicators and Rwanda the faster growth on both. Internet use in the two countries converged to about a third of the population by 2024, while GDP per capita in Kenya remained about twice Rwanda's.
['World Development Indicators, IT.NET.USER.ZS (ITU)', 'World Development Indicators, NY.GDP.PCAP.CD (World Bank national accounts)']


Validation is what makes the schema a contract. The cell below damages the JSON and shows the error that a downstream program would receive. Without validation it would receive a wrong value and no error.

In [8]:
from pydantic import ValidationError

damaged = completion.content.replace('"year": 2024', '"year": "twenty twenty-four"', 1)
try:
    w.ComparisonAnswer.model_validate_json(damaged)
except ValidationError as e:
    print(e.errors()[0]["loc"], "->", e.errors()[0]["msg"])

('countries', 0, 'internet_latest', 'year') -> Input should be a valid integer, unable to parse string as an integer


## 6. Live data and MCP

The same tool can read the World Bank API; by default it reads the committed file. The cell is guarded because the session must not depend on the network.

The Model Context Protocol (MCP) is a standard way to expose tools like this one to any client. The program's [Data360 MCP server](https://github.com/worldbank/data360-mcp) exposes search, metadata, and data tools over the World Bank's data platform; a desktop assistant, a chatbot, or this notebook can call them with the same schema idea as `get_indicator`. The slides cover it; the notebooks keep tools local so that they run offline.

In [9]:
try:
    live = w.get_indicator("IT.NET.USER.ZS", "Kenya", start=2022, live=True)
    print("live:", [(o["year"], round(o["value"], 1)) for o in live["observations"]])
except Exception as e:
    print("live call skipped:", type(e).__name__)

live: [(2022, 29.4), (2023, 32.1), (2024, 35.0)]


## Key points

- Numbers come from a tool over authoritative data; the model's memory is for language.
- A tool is a function with a description the model can read. The model fills the parameters; the application runs the function and returns the result.
- Absence of data is a tool result the application can test for.
- A schema turns the answer into a contract: validated on the way in, readable by other software on the way out.

## Exercise

Add a second tool that returns population (`SP.POP.TOTL`) and ask "How many people in Rwanda used the internet in 2024?". The answer needs two tool calls and a calculation; check whether the model does the arithmetic or asks for a tool to do it.

## Where this connects to the program

- [Practical Guide to Serving Official Statistics to AI Agents](https://worldbank.github.io/ai4data/cookbook/serving-statistics-to-agents/): the design of tools over a statistical catalog, in full.
- [Practical Guide to Evaluation Suites for Statistical AI](https://worldbank.github.io/ai4data/cookbook/evaluation-suites/): how to build and run the tests.
- [Proof-Carrying Numbers](https://worldbank.github.io/ai4data/docs/pcn/): the claim-tag protocol the verifier in this session follows.
- [Data360 MCP server](https://github.com/worldbank/data360-mcp): the tools of notebook 04, exposed to any MCP client over the World Bank's data platform.
- [End-to-end reference implementation](https://worldbank.github.io/ai4data/docs/reference-implementation/): documentation, validation, publication, agent access, and verification in one run.